In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(f"GROQ_API_KEY: {GROQ_API_KEY}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "explain {topic} in simple terms."),
])

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY
)

chain = prompt | llm

In [ ]:
result = chain.invoke({"topic": "AI"})

print(result.content)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
loader = TextLoader("hello.txt")

In [ ]:
doc_loader = loader.load()

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size = 100, chunk_overlap = 10)

In [ ]:
text_splitter.split_documents(doc_loader)

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
db = FAISS.from_documents(
    doc_loader,
    embedding_model
)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print(groq_api_key)

In [ ]:
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
retriever = db.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Answer the user's question using only the provided context.

If the answer is not present in the context, simply say:
"I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("What is my name ?")

print(response)